In [1]:
import pandas as pd
import math
from collections import Counter
import pprint

data = pd.read_csv("iris.csv")

# Discretizar atributos numéricos
def discretize(column):
    return pd.cut(column, bins=3, labels=['Baixo', 'Medio', 'Alto'])

numeric_columns = ['sepallength', 'sepalwidth', 'petallength', 'petalwidth']

for col in numeric_columns:
    data[col] = discretize(data[col])

# Entropia
def entropy(labels):
    total = len(labels)
    counts = Counter(labels)
    return -sum((count/total) * math.log2(count/total) for count in counts.values() if count > 0)

#Ganho de Informação
def information_gain(data, split_attribute, target_attribute='class'):
    total_entropy = entropy(data[target_attribute])
    values = data[split_attribute].unique()
    subset_entropy = 0.0
    for value in values:
        subset = data[data[split_attribute] == value]
        weight = len(subset) / len(data)
        subset_entropy += weight * entropy(subset[target_attribute])
    return total_entropy - subset_entropy

def id3(data, features, target_attribute='class'):
    labels = data[target_attribute]

    if len(set(labels)) == 1:
        return labels.iloc[0]

    # Caso 2: sem atributos para dividir
    if len(features) == 0:
        return labels.mode()[0]

    # Escolher feature com maior ganho de informação
    gains = [(attr, information_gain(data, attr, target_attribute)) for attr in features]
    best_attr = max(gains, key=lambda x: x[1])[0]

    tree = {best_attr: {}}
    for value in data[best_attr].unique():
        subset = data[data[best_attr] == value]
        if subset.empty:
            tree[best_attr][value] = labels.mode()[0]
        else:
            remaining_features = [f for f in features if f != best_attr]
            subtree = id3(subset, remaining_features, target_attribute)
            tree[best_attr][value] = subtree
    return tree

# === Função para classificar um exemplo usando a árvore gerada ===
def classify(tree, example):
    if not isinstance(tree, dict):
        return tree  # chegou a uma folha

    attribute = next(iter(tree))
    value = example[attribute]

    if value in tree[attribute]:
        return classify(tree[attribute][value], example)
    else:
        # caso o valor não esteja presente (pode ocorrer se houver valores fora do treino)
        return None


# === Rodar o ID3 ===
features = numeric_columns.copy()
decision_tree = id3(data, features)

# === Exibir árvore ===
pprint.pprint(decision_tree)

# === Prever e calcular a accuracy ===
correct = 0
total = len(data)

for i in range(total):
    example = data.iloc[i]
    true_label = example['class']
    predicted_label = classify(decision_tree, example)

    if predicted_label == true_label:
        correct += 1

accuracy = correct / total
print(f"\nAccuracy da Decision Tree (ID3): {accuracy:.2%}")




{'petalwidth': {'Alto': {'petallength': {'Alto': 'Iris-virginica',
                                         'Medio': {'sepalwidth': {'Baixo': 'Iris-virginica',
                                                                  'Medio': {'sepallength': {'Medio': 'Iris-virginica'}}}}}},
                'Baixo': 'Iris-setosa',
                'Medio': {'petallength': {'Alto': {'sepallength': {'Alto': 'Iris-virginica',
                                                                   'Medio': {'sepalwidth': {'Baixo': 'Iris-virginica',
                                                                                            'Medio': 'Iris-versicolor'}}}},
                                          'Medio': {'sepallength': {'Alto': 'Iris-versicolor',
                                                                    'Baixo': {'sepalwidth': {'Baixo': 'Iris-versicolor',
                                                                                             'Medio': 'Iris-versicolor'}},


In [4]:


# === Ler o dataset Connect 4 ===
data = pd.read_csv("connect4_mcts_dataset_clean.csv")

# === Atributos e alvo ===
features = [f'col_{i}' for i in range(7)]
target_attribute = 'suggested_move'

# === Entropia ===
def entropy(labels):
    total = len(labels)
    counts = Counter(labels)
    return -sum((count/total) * math.log2(count/total) for count in counts.values() if count > 0)

# === Ganho de informação ===
def information_gain(data, split_attribute, target_attribute='suggested_move'):
    total_entropy = entropy(data[target_attribute])
    values = data[split_attribute].unique()
    subset_entropy = 0.0
    for value in values:
        subset = data[data[split_attribute] == value]
        weight = len(subset) / len(data)
        subset_entropy += weight * entropy(subset[target_attribute])
    return total_entropy - subset_entropy

# === ID3 ===
def id3(data, features, target_attribute='suggested_move'):
    labels = data[target_attribute]

    if len(set(labels)) == 1:
        return labels.iloc[0]

    if len(features) == 0:
        return labels.mode()[0]

    gains = [(attr, information_gain(data, attr, target_attribute)) for attr in features]
    best_attr = max(gains, key=lambda x: x[1])[0]

    tree = {best_attr: {}}
    for value in data[best_attr].unique():
        subset = data[data[best_attr] == value]
        if subset.empty:
            tree[best_attr][value] = labels.mode()[0]
        else:
            remaining_features = [f for f in features if f != best_attr]
            subtree = id3(subset, remaining_features, target_attribute)
            tree[best_attr][value] = subtree
    return tree

# === Classificação de um exemplo ===
def classify(tree, example):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = example[attribute]

    if value in tree[attribute]:
        return classify(tree[attribute][value], example)
    else:
        return None

# === Treinar e avaliar ===
decision_tree = id3(data, features, target_attribute)

pprint.pprint(decision_tree)

correct = 0
total = len(data)

for i in range(total):
    example = data.iloc[i]
    true_label = example[target_attribute]
    predicted_label = classify(decision_tree, example)
    if predicted_label == true_label:
        correct += 1

accuracy = correct / total
print(f"\nAccuracy da Decision Tree (ID3): {accuracy:.2%}")


{'col_3': {'bbbbbb': {'col_4': {'bbbbbb': {'col_0': {'bbbbbb': {'col_2': {'bbbbbb': {'col_1': {'bbbbbb': {'col_5': {'bbbbbb': {'col_6': {'xbbbbb': 1}},
                                                                                                                    'xbbbbb': {'col_6': {'bbbbbb': 5}}}},
                                                                                               'obbbbb': {'col_5': {'bbbbbb': 3,
                                                                                                                    'xbbbbb': 1}},
                                                                                               'xbbbbb': {'col_5': {'bbbbbb': {'col_6': {'bbbbbb': 1}},
                                                                                                                    'xobbbb': {'col_6': {'bbbbbb': 2}}}},
                                                                                               'xobbbb': {'col_5': {'bbbbbb': {'